# Week 8: Sequence Representations and Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/08/Week_08_Sequence_Representations_Attention.ipynb)

**Course:** Neural Architectures and Representation Learning (Master level)


## Learning goals

By the end of this session, you should be able to:

- Explain why order matters in sequence data.
- Describe embeddings as learned vector representations of symbols or words.
- Train a small character-level RNN.
- Inspect hidden states as evolving sequence memory.
- Visualize learned word embeddings.
- Explain attention as selective focus over sequence elements.
- Interpret attention heatmaps carefully, without treating them as perfect explanations.

**Course habit:** change one thing -> run -> observe -> explain.

**New representation habit:** inspect how the representation changes as context grows.


---

## Environment

**Dependencies:** `torch`, `numpy`, `matplotlib`, `scikit-learn`. CPU is enough for the full notebook.

### Local (uv)

From the repo root:

```bash
uv sync
uv run jupyter notebook weeks/08/Week_08_Sequence_Representations_Attention.ipynb
```

### Colab

1. Open the notebook via the badge above.
2. Runtime -> Change runtime type -> CPU is fine.
3. Run the setup cell below.

This notebook uses tiny inline datasets. There are no dataset downloads.


In [ ]:
import math
import random
import re
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

print("torch:", torch.__version__)
print("numpy:", np.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(8)


---

## 1. From spatial representations to sequence representations

Last week, a CNN turned pixels into feature maps and then into a reusable visual representation.

This week, the input is not a grid. It is a sequence:

```text
characters -> name
words -> sentence
measurements over time -> signal
clicks over time -> user path
```

A sequence representation has to answer a different question:

> What should I remember from the earlier tokens when I process the current token?

**Pause and predict**

1. Why do `dog bites man` and `man bites dog` have different meanings?
2. What information is lost if we sort the words alphabetically?
3. Why might a final prediction depend on a word that appeared much earlier?


### External anchors for this week

These are optional references, not required dependencies.

| Chapter | Useful links |
|---|---|
| Recurrent memory | [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/), [The Unreasonable Effectiveness of RNNs](https://karpathy.github.io/2015/05/21/rnn-effectiveness/), [PyTorch character-level RNN tutorial](https://docs.pytorch.org/tutorials/intermediate/char_rnn_classification_tutorial.html) |
| Embeddings | [3Blue1Brown embeddings video](https://www.youtube.com/watch?v=wjZofJX0v4M&list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi&index=6&t=747s), [TensorFlow Embedding Projector](https://projector.tensorflow.org/), [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/) |
| Attention | [Visualizing seq2seq attention](https://jalammar.github.io/visualizing-neural-machine-translation-mechanics-of-seq2seq-models-with-attention/), [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/), [Attention Is All You Need](https://arxiv.org/abs/1706.03762) |

**RNN vs LSTM note:** `RNN = simple recurrent memory`; `LSTM = recurrent memory with learned gates`. We use a plain RNN in code because it is easiest to inspect; the LSTM link explains why gated memory was introduced for longer or harder sequences.


---

## 2. Why order matters

A bag-of-words view remembers which tokens appeared, but not where they appeared.

That is sometimes enough. It is often not enough.


In [ ]:
examples = [
    "dog bites man",
    "man bites dog",
    "not good",
    "good not",
    "the patient improved after treatment",
    "the treatment improved after patient",
]

for sentence in examples:
    tokens = sentence.split()
    bag = sorted(tokens)
    print(f"sequence: {sentence:38s}  bag-of-words view: {bag}")


In [ ]:
def draw_sequence(tokens, title="Sequence read left to right"):
    fig, ax = plt.subplots(figsize=(max(6, len(tokens) * 1.2), 1.8))
    ax.set_title(title)
    ax.set_xlim(-0.5, len(tokens) - 0.5)
    ax.set_ylim(-0.5, 0.8)
    ax.axis("off")
    for i, tok in enumerate(tokens):
        ax.text(i, 0, tok, ha="center", va="center", fontsize=13,
                bbox=dict(boxstyle="round,pad=0.35", facecolor="#e8f1ff", edgecolor="#4c78a8"))
        if i < len(tokens) - 1:
            ax.annotate("", xy=(i + 0.72, 0), xytext=(i + 0.28, 0),
                        arrowprops=dict(arrowstyle="->", color="#444", lw=1.5))
    plt.show()

draw_sequence("the model reads one token at a time".split())


---

## 3. Character-level sequence modeling

We will use a tiny names dataset. The task is to classify a surname into a broad language/origin group.

This is a toy pattern-learning task. It is useful because:

- the vocabulary is tiny: characters
- sequences have variable length
- suffixes and character patterns matter
- the model trains quickly

It is **not** a serious demographic classifier.

### Colab pointers for the RNN section

- Use the default CPU runtime; this section is intentionally small enough for Colab without GPU.
- Run cells in order, because later inspection cells reuse the trained `char_model`.
- The main object to inspect is `CharRNNClassifier`: `embedding` turns characters into vectors, `rnn` updates memory, and `classifier` maps the final memory to a label.
- The useful knobs are `embedding_dim`, `hidden_dim`, `epochs`, and `lr`. Change one or two, then compare the hidden-state and prefix-probability plots.
- If Colab state gets confusing, use `Runtime -> Restart session and run all`.


In [ ]:
raw_names = {
    "English": [
        "Smith", "Johnson", "Williams", "Brown", "Jones", "Anderson", "Wilson", "Taylor",
        "Thomas", "Moore", "Martin", "Jackson", "Thompson", "White", "Harris", "Clark",
        "Lewis", "Walker", "Hall", "Allen", "Young", "King", "Wright", "Scott",
    ],
    "Italian": [
        "Rossi", "Russo", "Ferrari", "Esposito", "Bianchi", "Romano", "Colombo", "Ricci",
        "Marino", "Greco", "Bruno", "Gallo", "Conti", "Deluca", "Costa", "Mancini",
        "Rizzo", "Lombardi", "Moretti", "Barbieri", "Fontana", "Santoro", "Longo", "Leone",
    ],
    "German": [
        "Muller", "Schmidt", "Schneider", "Fischer", "Weber", "Meyer", "Wagner", "Becker",
        "Schulz", "Hoffmann", "Schafer", "Koch", "Bauer", "Richter", "Klein", "Wolf",
        "Schroder", "Neumann", "Schwarz", "Zimmermann", "Braun", "Kruger", "Hartmann", "Lange",
    ],
    "Slavic": [
        "Novak", "Kowalski", "Wisniewski", "Kaminski", "Lewandowski", "Zielinski", "Sokolov", "Popov",
        "Ivanov", "Petrov", "Volkov", "Morozov", "Smirnov", "Kuznetsov", "Horvat", "Kovacic",
        "Jankovic", "Markovic", "Pavlov", "Dvorak", "Kral", "Vesely", "Nowak", "Mazur",
    ],
}

labels = list(raw_names.keys())
label_to_idx = {label: i for i, label in enumerate(labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

all_examples = []
for label, names in raw_names.items():
    for name in names:
        all_examples.append((name.lower(), label_to_idx[label]))

random.shuffle(all_examples)
split = int(0.8 * len(all_examples))
train_examples = all_examples[:split]
test_examples = all_examples[split:]

chars = sorted(set("".join(name for name, _ in all_examples)))
char_to_idx = {"<PAD>": 0}
for ch in chars:
    char_to_idx[ch] = len(char_to_idx)
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

print("examples:", len(all_examples))
print("labels:", labels)
print("character vocabulary:", "".join(chars))
print("train/test:", len(train_examples), len(test_examples))


In [ ]:
def encode_name(name):
    return torch.tensor([char_to_idx[ch] for ch in name.lower()], dtype=torch.long)


def collate_names(batch):
    encoded = [encode_name(name) for name, _ in batch]
    lengths = torch.tensor([len(x) for x in encoded], dtype=torch.long)
    y = torch.tensor([label for _, label in batch], dtype=torch.long)
    max_len = int(lengths.max())
    x = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, item in enumerate(encoded):
        x[i, : len(item)] = item
    return x, lengths, y

train_loader = DataLoader(train_examples, batch_size=16, shuffle=True, collate_fn=collate_names)
test_loader = DataLoader(test_examples, batch_size=32, shuffle=False, collate_fn=collate_names)

lengths_by_label = defaultdict(list)
for name, y in all_examples:
    lengths_by_label[idx_to_label[y]].append(len(name))

plt.figure(figsize=(7, 3))
plt.bar(lengths_by_label.keys(), [np.mean(v) for v in lengths_by_label.values()], color="#4c78a8")
plt.ylabel("Average name length")
plt.title("Tiny names dataset")
plt.show()


### What the model will learn

The model reads one character at a time.

This notebook uses a plain RNN: simple recurrent memory. An LSTM is the same broad recurrent idea, but with learned gates that decide what to remember, forget, write, and expose.

At each step:

```text
current character + previous hidden state -> new hidden state
```

The hidden state is the model's evolving memory of the prefix it has read so far.


In [ ]:
class CharRNNClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, embedding_dim=12, hidden_dim=24):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h_n = self.rnn(packed)
        final_hidden = h_n[-1]
        logits = self.classifier(final_hidden)
        return logits

    @torch.no_grad()
    def hidden_path(self, name):
        self.eval()
        x = encode_name(name).unsqueeze(0).to(next(self.parameters()).device)
        embedded = self.embedding(x)
        outputs, _ = self.rnn(embedded)
        logits_each_step = self.classifier(outputs.squeeze(0))
        return outputs.squeeze(0).cpu(), logits_each_step.cpu()


def accuracy_from_logits(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def train_char_rnn(embedding_dim=12, hidden_dim=24, epochs=45, lr=0.02):
    set_seed(8)
    model = CharRNNClassifier(
        vocab_size=len(char_to_idx),
        num_classes=len(labels),
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "train_acc": [], "test_acc": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_acc = 0.0
        n_batches = 0
        for x, lengths, y in train_loader:
            x, lengths, y = x.to(device), lengths.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x, lengths)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            total_acc += accuracy_from_logits(logits, y)
            n_batches += 1

        model.eval()
        test_accs = []
        with torch.no_grad():
            for x, lengths, y in test_loader:
                x, lengths, y = x.to(device), lengths.to(device), y.to(device)
                test_accs.append(accuracy_from_logits(model(x, lengths), y))

        history["loss"].append(total_loss / n_batches)
        history["train_acc"].append(total_acc / n_batches)
        history["test_acc"].append(float(np.mean(test_accs)))

    return model, history


def plot_char_history(history, title="Character RNN training"):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(history["loss"])
    axes[0].set_title("Training loss")
    axes[0].set_xlabel("epoch")
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["test_acc"], label="test")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylim(0, 1.05)
    axes[1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

char_model, char_history = train_char_rnn()
plot_char_history(char_history)
print("final test accuracy:", round(char_history["test_acc"][-1], 3))


In [ ]:
@torch.no_grad()
def predict_names(model, examples):
    model.eval()
    rows = []
    for name, y in examples:
        x, lengths, label = collate_names([(name, y)])
        logits = model(x.to(device), lengths.to(device))
        probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
        pred = int(probs.argmax())
        rows.append((name, idx_to_label[y], idx_to_label[pred], float(probs[pred])))
    return rows

rows = predict_names(char_model, test_examples[:12])
for name, true_label, pred_label, confidence in rows:
    print(f"{name:14s} true={true_label:8s} pred={pred_label:8s} confidence={confidence:.2f}")


In [ ]:
@torch.no_grad()
def plot_confusion(model, examples):
    y_true, y_pred = [], []
    for name, y in examples:
        x, lengths, label = collate_names([(name, y)])
        logits = model(x.to(device), lengths.to(device))
        y_true.append(y)
        y_pred.append(int(logits.argmax(dim=1).item()))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels=labels, rotation=30, ha="right")
    ax.set_yticks(range(len(labels)), labels=labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Test confusion matrix")
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="#222")
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

plot_confusion(char_model, test_examples)


---

## 4. Hidden states as evolving memory

A final prediction hides the interesting part.

For a sequence model, we can inspect the hidden state after each character. This shows how the representation changes as the name unfolds.


In [ ]:
def plot_hidden_path(model, name, max_units=16):
    hidden, logits_each = model.hidden_path(name.lower())
    hidden = hidden[:, :max_units].numpy().T
    chars_for_plot = list(name.lower())

    fig, ax = plt.subplots(figsize=(max(6, len(chars_for_plot) * 0.7), 4))
    im = ax.imshow(hidden, aspect="auto", cmap="coolwarm")
    ax.set_xticks(range(len(chars_for_plot)), labels=chars_for_plot)
    ax.set_yticks(range(hidden.shape[0]), labels=[f"h{i}" for i in range(hidden.shape[0])])
    ax.set_xlabel("character position")
    ax.set_ylabel("hidden unit")
    ax.set_title(f"Hidden-state path for '{name}'")
    fig.colorbar(im, ax=ax, fraction=0.025)
    plt.tight_layout()
    plt.show()


def plot_prefix_predictions(model, name):
    _, logits_each = model.hidden_path(name.lower())
    probs = F.softmax(logits_each, dim=1).numpy()
    prefixes = [name[: i + 1].lower() for i in range(len(name))]

    plt.figure(figsize=(max(7, len(name) * 0.75), 3.5))
    for i, label in enumerate(labels):
        plt.plot(prefixes, probs[:, i], marker="o", label=label)
    plt.ylim(0, 1.05)
    plt.ylabel("predicted probability")
    plt.xlabel("prefix read so far")
    plt.title(f"Prediction evolves while reading '{name}'")
    plt.legend(ncol=2)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

inspect_name = "Kowalski"
plot_hidden_path(char_model, inspect_name)
plot_prefix_predictions(char_model, inspect_name)


**Pause and reflect**

1. At which character does the model become confident?
2. Do any hidden units change sharply near a suffix?
3. Is the final prediction based on one character, or on a pattern across time?


---

## 5. Coding block 1: change the sequence model

**Goal:** edit the model settings, train briefly, and inspect the representation path.

**Core path**

1. Change `embedding_dim`, `hidden_dim`, or `epochs`.
2. Re-run training.
3. Inspect the hidden-state heatmap and prefix predictions.
4. Explain what changed.

Keep the run short. The point is comparison, not perfect accuracy.


In [ ]:
# TODO: edit one or two values, then re-run.
student_sequence_config = {
    "embedding_dim": 10,
    "hidden_dim": 18,
    "epochs": 35,
    "lr": 0.02,
}

student_char_model, student_char_history = train_char_rnn(**student_sequence_config)
plot_char_history(student_char_history, "Student character-RNN experiment")

student_inspect_name = "Schneider"  # TODO: try another name from the dataset
plot_hidden_path(student_char_model, student_inspect_name)
plot_prefix_predictions(student_char_model, student_inspect_name)


### Sequence experiment report prompt

Write 4-6 sentences:

1. What did you change?
2. What happened to train/test accuracy?
3. What changed in the hidden-state heatmap?
4. Did the model become confident earlier or later in the sequence?
5. What does this suggest about the representation?


---

## 6. From character embeddings to word embeddings

Now we move from characters to words.

An embedding layer stores a learned vector for each token. During training, words that help the model make similar decisions often move into similar parts of the embedding space.

Important caution: with a tiny dataset, the embeddings are not rich language understanding. They are a small, inspectable version of the mechanism.

### Colab pointers for the embedding section

- The embedding table lives in `mean_model.embedding.weight` after training. Each row is one learned word vector.
- The PCA plot is a compressed 2D view of those vectors; it is useful for intuition, not a final proof of meaning.
- `nearest_words(...)` uses cosine similarity in the learned embedding space. Try words that appear in the tiny vocabulary printed above.
- Custom attention sentences should mostly use known vocabulary words; unseen words become `<UNK>`, which makes interpretation weaker.
- If you edit the sentence dataset, re-run the vocabulary, dataloader, and training cells below it.


In [ ]:
sentence_data = [
    ("i loved the movie", 1),
    ("the film was excellent", 1),
    ("what a great story", 1),
    ("this product feels amazing", 1),
    ("the service was wonderful", 1),
    ("a pleasant and charming film", 1),
    ("the lecture was clear", 1),
    ("i enjoyed the concert", 1),
    ("the meal tasted fantastic", 1),
    ("the app is useful", 1),
    ("not bad at all", 1),
    ("the update is surprisingly good", 1),
    ("i hated the movie", 0),
    ("the film was terrible", 0),
    ("what a boring story", 0),
    ("this product feels awful", 0),
    ("the service was poor", 0),
    ("a dull and confusing film", 0),
    ("the lecture was unclear", 0),
    ("i disliked the concert", 0),
    ("the meal tasted bad", 0),
    ("the app is useless", 0),
    ("not good at all", 0),
    ("the update is surprisingly broken", 0),
    ("good clear useful service", 1),
    ("excellent pleasant amazing movie", 1),
    ("bad dull useless service", 0),
    ("terrible awful boring movie", 0),
]

random.shuffle(sentence_data)

TOKEN_RE = re.compile(r"[a-z]+")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

counter = Counter()
for text, _ in sentence_data:
    counter.update(tokenize(text))

word_to_idx = {"<PAD>": 0, "<UNK>": 1}
for word in sorted(counter):
    word_to_idx[word] = len(word_to_idx)
idx_to_word = {i: word for word, i in word_to_idx.items()}

print("sentences:", len(sentence_data))
print("vocabulary size:", len(word_to_idx))
print("sample tokens:", tokenize(sentence_data[0][0]))


In [ ]:
def encode_sentence(text):
    ids = [word_to_idx.get(tok, word_to_idx["<UNK>"]) for tok in tokenize(text)]
    return torch.tensor(ids, dtype=torch.long)


def collate_sentences(batch):
    encoded = [encode_sentence(text) for text, _ in batch]
    lengths = torch.tensor([len(x) for x in encoded], dtype=torch.long)
    y = torch.tensor([label for _, label in batch], dtype=torch.long)
    max_len = int(lengths.max())
    x = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, item in enumerate(encoded):
        x[i, : len(item)] = item
    return x, lengths, y

text_loader = DataLoader(sentence_data, batch_size=8, shuffle=True, collate_fn=collate_sentences)

class MeanEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=10):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.classifier = nn.Linear(embedding_dim, 2)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        mask = (x != 0).unsqueeze(-1)
        summed = (emb * mask).sum(dim=1)
        mean = summed / lengths.clamp_min(1).unsqueeze(1)
        logits = self.classifier(mean)
        return logits


def train_mean_embedding_model(embedding_dim=10, epochs=140, lr=0.03):
    set_seed(9)
    model = MeanEmbeddingClassifier(len(word_to_idx), embedding_dim=embedding_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        losses, accs = [], []
        model.train()
        for x, lengths, y in text_loader:
            x, lengths, y = x.to(device), lengths.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x, lengths)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            accs.append(accuracy_from_logits(logits, y))
        history.append((float(np.mean(losses)), float(np.mean(accs))))
    return model, history

mean_model, mean_history = train_mean_embedding_model()

plt.figure(figsize=(6, 3))
plt.plot([x[0] for x in mean_history], label="loss")
plt.plot([x[1] for x in mean_history], label="accuracy")
plt.title("Mean-embedding classifier")
plt.xlabel("epoch")
plt.legend()
plt.tight_layout()
plt.show()
print("final training accuracy:", round(mean_history[-1][1], 3))


In [ ]:
def plot_word_embeddings(model, words_to_show=None):
    model.eval()
    with torch.no_grad():
        weights = model.embedding.weight.detach().cpu().numpy()
    ids = [i for i in range(2, len(idx_to_word))]
    words = [idx_to_word[i] for i in ids]
    vectors = weights[ids]
    coords = PCA(n_components=2, random_state=0).fit_transform(vectors)

    if words_to_show is not None:
        keep = [i for i, w in enumerate(words) if w in set(words_to_show)]
    else:
        keep = list(range(len(words)))

    plt.figure(figsize=(8, 6))
    plt.scatter(coords[keep, 0], coords[keep, 1], s=35, color="#4c78a8")
    for i in keep:
        plt.text(coords[i, 0] + 0.02, coords[i, 1] + 0.02, words[i], fontsize=9)
    plt.title("2D view of learned word embeddings (PCA)")
    plt.xlabel("component 1")
    plt.ylabel("component 2")
    plt.tight_layout()
    plt.show()

interesting_words = [
    "good", "great", "excellent", "amazing", "useful", "clear",
    "bad", "terrible", "awful", "boring", "useless", "unclear",
    "movie", "film", "service", "lecture", "not",
]
plot_word_embeddings(mean_model, interesting_words)


In [ ]:
def nearest_words(model, query_word, k=6):
    if query_word not in word_to_idx:
        print(f"'{query_word}' is not in the tiny vocabulary.")
        return
    with torch.no_grad():
        W = model.embedding.weight.detach().cpu()
        q = W[word_to_idx[query_word]]
        sims = F.cosine_similarity(q.unsqueeze(0), W, dim=1)
    candidates = []
    for idx, score in enumerate(sims.tolist()):
        word = idx_to_word[idx]
        if word not in {"<PAD>", "<UNK>", query_word}:
            candidates.append((word, score))
    candidates.sort(key=lambda x: x[1], reverse=True)
    print(f"Nearest words to '{query_word}':")
    for word, score in candidates[:k]:
        print(f"  {word:14s} cosine={score:.2f}")

nearest_words(mean_model, "good")
nearest_words(mean_model, "terrible")


**Pause and reflect**

1. Which nearby words make intuitive sense?
2. Which nearby words are surprising?
3. Why should we be careful when interpreting a 2D projection from a tiny dataset?


---

## 7. Attention intuition

Mean pooling gives every word equal weight.

Attention pooling learns a weight for each token, then builds a weighted sentence representation.

```text
word embeddings -> attention scores -> attention weights -> weighted representation -> prediction
```

The weights can be visualized as a heatmap. Treat them as useful evidence, not as a perfect explanation.

### What the heatmap is showing

For each word, the model computes a small score from that word embedding. A softmax turns those scores into attention weights that sum to 1. The sentence representation is then a weighted average of the word embeddings.

```text
attention weight near 0   -> this token contributes little to the pooled representation
attention weight near 1   -> this token dominates the pooled representation
```

In this notebook, attention is **single-head attention pooling**. It is much simpler than transformer self-attention. It does not compare every word with every other word; it only learns which tokens to emphasize before classification.

### How to read surprising attention

If the model attends to `excellent` in `the film was excellent`, that is intuitive. If it attends to `was` in `the film was terrible`, or treats `not good` and `not bad` similarly, that is a useful warning: the model may be exploiting tiny-dataset shortcuts rather than learning the rule we expected.

So the class interpretation should be:

- attention weights show what this trained model emphasized
- they can help us debug representation behavior
- they are not guaranteed to match human explanations
- negation usually needs more data, better architecture, or richer contextual modeling


In [ ]:
class AttentionPoolingClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=12):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.attention_score = nn.Linear(embedding_dim, 1)
        self.classifier = nn.Linear(embedding_dim, 2)

    def forward(self, x, lengths, return_attention=False):
        emb = self.embedding(x)
        scores = self.attention_score(torch.tanh(emb)).squeeze(-1)
        scores = scores.masked_fill(x == 0, -1e9)
        weights = F.softmax(scores, dim=1)
        context = (emb * weights.unsqueeze(-1)).sum(dim=1)
        logits = self.classifier(context)
        if return_attention:
            return logits, weights
        return logits


def train_attention_model(embedding_dim=12, epochs=170, lr=0.025):
    set_seed(10)
    model = AttentionPoolingClassifier(len(word_to_idx), embedding_dim=embedding_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        losses, accs = [], []
        model.train()
        for x, lengths, y in text_loader:
            x, lengths, y = x.to(device), lengths.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x, lengths)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            accs.append(accuracy_from_logits(logits, y))
        history.append((float(np.mean(losses)), float(np.mean(accs))))
    return model, history

attention_model, attention_history = train_attention_model()

plt.figure(figsize=(6, 3))
plt.plot([x[0] for x in attention_history], label="loss")
plt.plot([x[1] for x in attention_history], label="accuracy")
plt.title("Attention-pooling classifier")
plt.xlabel("epoch")
plt.legend()
plt.tight_layout()
plt.show()
print("final training accuracy:", round(attention_history[-1][1], 3))


In [ ]:
@torch.no_grad()
def predict_sentence(model, sentence):
    model.eval()
    x, lengths, y = collate_sentences([(sentence, 0)])
    x, lengths = x.to(device), lengths.to(device)
    if isinstance(model, AttentionPoolingClassifier):
        logits, weights = model(x, lengths, return_attention=True)
        weights = weights.squeeze(0).cpu().numpy()[: int(lengths.item())]
    else:
        logits = model(x, lengths)
        weights = None
    probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    pred = int(probs.argmax())
    return idx_to_sentiment[pred], probs, weights

idx_to_sentiment = {0: "negative", 1: "positive"}


def plot_attention(sentence, model=attention_model):
    tokens = tokenize(sentence)
    pred, probs, weights = predict_sentence(model, sentence)
    fig, ax = plt.subplots(figsize=(max(6, len(tokens) * 0.8), 1.8))
    ax.imshow(weights.reshape(1, -1), cmap="YlOrRd", aspect="auto", vmin=0, vmax=max(0.45, float(weights.max())))
    ax.set_xticks(range(len(tokens)), labels=tokens, rotation=30, ha="right")
    ax.set_yticks([])
    ax.set_title(f"Attention weights | prediction: {pred} | p(pos)={probs[1]:.2f}")
    for i, w in enumerate(weights):
        ax.text(i, 0, f"{w:.2f}", ha="center", va="center", color="#111")
    plt.tight_layout()
    plt.show()

for sentence in [
    "the film was excellent",
    "the film was terrible",
    "not good at all",
    "not bad at all",
]:
    plot_attention(sentence)


**Pause and reflect**

1. Which words receive high attention?
2. Does the attention pattern match your intuition?
3. Can attention be useful even when it is not a complete explanation?
4. For the surprising examples, what shortcut might the tiny model be using?


---

## 8. Coding block 2: embeddings and attention

**Goal:** inspect word representations and attention behavior.

**Core path**

1. Choose two query words and inspect nearest neighbors.
2. Try two custom sentences using words from the tiny vocabulary.
3. Compare the attention heatmaps.
4. Explain what the model seems to focus on.

Try to create at least one easy example and one ambiguous example.


In [ ]:
# TODO: change these words.
for query_word in ["excellent", "awful"]:
    nearest_words(mean_model, query_word, k=6)

# TODO: change these sentences, using words from the tiny vocabulary above.
student_sentences = [
    "the movie was good",
    "the movie was not good",
    "the service was surprisingly awful",
]

for sentence in student_sentences:
    plot_attention(sentence)


### Attention report prompt

Write 4-6 sentences:

1. Which nearest-neighbor results made sense?
2. Which attention heatmap was easiest to interpret?
3. Did the model handle negation or mixed evidence well?
4. What does the attention view reveal about the representation?
5. Which example looked misleading, and why?
6. What is one limitation of this tiny experiment?


---

## 9. Assignment 3 preview

Assignment 3 will ask you to analyze sequence representations.

Minimum evidence will likely include:

- a controlled sequence-model experiment
- hidden-state or embedding visualization
- attention visualization or token-importance analysis
- examples where the model behaves well
- examples where the model behaves surprisingly
- a short explanation of what the representation captures and misses

The main question is not "did the model get a high score?" The main question is: **what information did the learned representation make visible or usable?**


---

## Wrap-up: takeaways

1. Sequence models care about order and context.
2. Embeddings turn discrete symbols into learned vectors.
3. Hidden states are evolving representations of the prefix seen so far.
4. Word embeddings can be inspected with neighbors and 2D projections.
5. Attention learns selective weights over tokens.
6. Attention is a bridge to transformers, but full transformer architecture is Week 9.


---

## Homework / Post-class Extensions

Optional unless assigned.

| Idea | What to try |
|------|-------------|
| More names | Add 5-10 names per class and re-run the RNN |
| Earlier confidence | Compare prefix probabilities for short and long names |
| Hidden-unit stories | Pick one hidden unit and track it across several names |
| Mean vs attention | Compare predictions from mean pooling and attention pooling |
| Negation stress test | Add more `not ...` examples and inspect attention |
| Larger embeddings | Open TensorFlow Embedding Projector and compare with this tiny PCA view |
